urllib用于实现HTTP请求的发送，而不需要关心HTTP协议本身甚至更底层的实现。同时还会把服务器的Response转化为Python对象。
urllib是Python的内置库，不需要额外安装。
包含下面四个模块：
1. request：这是基本的HTTP请求模块，可以模拟请求的发送。
2. error：异常处理模块。
3. parse：一个工具模块。日工了许多URL的处理方法，如拆分，解析，合并等。
4. robotparser：主要用来识别网站的robot.txt

## 1. urlopen
urllib.request提供了最基本的构造HTTP请求的方法，利用这个模块可以模拟浏览器的请求发起过程，同时他还具有处理Authentication，Redirection，浏览器Cookie以及其他的一些功能。
下面是一个简单示例：

In [5]:
import urllib.request
import pprint as pp

response = urllib.request.urlopen("https://www.python.org/")
print(response.status)
print(type(response))
pp.pprint(response.getheaders())

200
<class 'http.client.HTTPResponse'>
[('Connection', 'close'),
 ('Content-Length', '52294'),
 ('x-frame-options', 'SAMEORIGIN'),
 ('server', 'nginx'),
 ('via', '1.1 varnish, 1.1 varnish, 1.1 varnish'),
 ('content-type', 'text/html; charset=utf-8'),
 ('Accept-Ranges', 'bytes'),
 ('Date', 'Sun, 07 Jun 2026 13:05:57 GMT'),
 ('Age', '1772'),
 ('X-Served-By',
  'cache-iad-kiad7000081-IAD, cache-iad-kiad7000081-IAD, '
  'cache-iad-kiad7000081-IAD, cache-nrt-rjaa8190034-NRT'),
 ('X-Cache', 'MISS, HIT, HIT'),
 ('X-Cache-Hits', '0, 412, 26'),
 ('X-Timer', 'S1780837557.014591,VS0,VE0'),
 ('Vary', 'Cookie'),
 ('Strict-Transport-Security', 'max-age=63072000; includeSubDomains; preload')]


由于python官网的网页返回的是一个二进制数据，这里就不打印具体内容了。总之，我们成功拿到了官网的源代码。
可以看到，返回值是一个http.client.HTTPResponse类型。
主要包含了下列方法
- read
- readinto
- getheader
- getheaders
- fileno
以及下列属性
- msg
- version
- status
- reason
- debuglevel
- closed

In [6]:
print(response.getheader("Server"))

nginx


我们还可以传递别的参数，其API如下
urllib.request.urlopen(url, data=None, [timeout,]*, cafile=None, capath=None, cadefault=False, context=None)
1. data
添加此参数时，需要使用bytes方法将参数转化为字节流编码格式内容，即bytes类型。另外，如果传递了data，其请求方式就不再是GET而是POST了。
我们用https://httpbin.org来测试一下。这个网站可以提供HTTP请求测试，他会回显输入的一些信息。

In [7]:
import urllib.parse
import urllib.request

data = bytes(urllib.parse.urlencode({"name": "germy"}), encoding="utf-8")
response = urllib.request.urlopen("https://httpbin.org/post", data=data)
print(response.read().decode("utf-8"))

{
  "args": {}, 
  "data": "", 
  "files": {}, 
  "form": {
    "name": "germy"
  }, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Content-Length": "10", 
    "Content-Type": "application/x-www-form-urlencoded", 
    "Host": "httpbin.org", 
    "User-Agent": "Python-urllib/3.12", 
    "X-Amzn-Trace-Id": "Root=1-6a256e54-7ff4ae03398553a6789db236"
  }, 
  "json": null, 
  "origin": "217.178.17.255", 
  "url": "https://httpbin.org/post"
}



可以看到，我们传入的data出现在了form中，表明这是一个表单提交，我们的data被当做表单内容了。

2. timeout
这个参数控制超时，单位为秒。如果超过了这个时间还没有得到返回值，就会抛出异常。如果不指定的话，默认使用一个全局的超时时间。

In [10]:
import urllib.request, urllib.error

try:
    response = urllib.request.urlopen("https://www.httpbin.org/get", timeout=0.01)
except urllib.error.URLError as e:
    print(e.reason)

timed out


通过设置一个非常短的超时，我们可以看到触发了URLError。

3. 其它参数
context参数，只能是ssl.SSLContext类型，用来指定SSL的设置。
ca系列参数则是控制CA证书极其路径，在HTTPS请求时可能会用上
**cadefault已经被弃用**

## 2. Request
urlopen可以发起比较基本和简单的请求，但是当我们需要更多操作，比如添加headers时，就需要用到Request类了。
其API是：
class urllib.request.Request(url, data=None, headers={}, origin_req_host=None, unverifilable=False, method=None)
- url和data和urlopen的相同。
- headers可以在构造函数里添加，也可以通过add_header来添加
- origin_req_host指的是请求方的host或者IP
- unverifiable表示请求是否无法验证，意思是用户没有足够的权限来接受这个请求的结果，比如请求一个HTML文档中的图片，但是没有自动抓取图像的权限，此时就是True。
- method指定GET，POST等方法

In [11]:
from urllib import request, parse

url = "https://httpbin.org/post"
headers = {
    "User-Agent": "Mozilla/4.0 (compatible; MSIE 5.5; Windows NT)",
    "Host": "www.httpbin.org",
}

data = bytes(parse.urlencode({"name": "germy"}), encoding="utf-8")
req = request.Request(url=url, data=data, headers=headers, method="POST")
response = request.urlopen(req)
print(response.read().decode("utf-8"))

{
  "args": {}, 
  "data": "", 
  "files": {}, 
  "form": {
    "name": "germy"
  }, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Content-Length": "10", 
    "Content-Type": "application/x-www-form-urlencoded", 
    "Host": "www.httpbin.org", 
    "User-Agent": "Mozilla/4.0 (compatible; MSIE 5.5; Windows NT)", 
    "X-Amzn-Trace-Id": "Root=1-6a2574b4-74d762943cc7584a73aa43f0"
  }, 
  "json": null, 
  "origin": "217.178.17.255", 
  "url": "https://www.httpbin.org/post"
}

